# 00 — Fetch the EEG eye-state dataset

This notebook downloads the UCI EEG Eye State dataset (repository
id `264`), asserts the expected shape and label distribution, writes
the canonical raw CSV to `data/raw/eeg_eye_state.csv`, and records a
SHA-256 manifest. It is idempotent: if the CSV already exists locally
the download step is skipped.

Source: <https://archive.ics.uci.edu/dataset/264/eeg+eye+state>


In [1]:
import hashlib, json, os, sys
from datetime import datetime, timezone

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

RAW_DIR = os.path.join(REPO_ROOT, 'data', 'raw')
RAW_CSV = os.path.join(RAW_DIR, 'eeg_eye_state.csv')
MANIFEST = os.path.join(RAW_DIR, 'manifest.json')
os.makedirs(RAW_DIR, exist_ok=True)
RAW_CSV, MANIFEST

('/home/ubuntu/repos/cogs109-eeg-eye-state-classifier/data/raw/eeg_eye_state.csv',
 '/home/ubuntu/repos/cogs109-eeg-eye-state-classifier/data/raw/manifest.json')

## Load (or skip if already on disk)

In [2]:
if os.path.exists(RAW_CSV):
    import pandas as pd
    df = pd.read_csv(RAW_CSV)
    print(f'Loaded cached raw CSV: {RAW_CSV}  shape={df.shape}')
else:
    from ucimlrepo import fetch_ucirepo
    import pandas as pd
    repo = fetch_ucirepo(id=264)
    df = pd.concat([repo.data.features, repo.data.targets], axis=1)
    df.to_csv(RAW_CSV, index=False)
    print(f'Downloaded UCI #264 and wrote {RAW_CSV}  shape={df.shape}')


Loaded cached raw CSV: /home/ubuntu/repos/cogs109-eeg-eye-state-classifier/data/raw/eeg_eye_state.csv  shape=(14980, 15)


## Verify shape, columns, class counts, and NaN-freeness

In [3]:
from src.data import CHANNELS, LABEL_COL

assert df.shape == (14980, 15), f'unexpected shape {df.shape}'
assert list(df.columns[:14]) == list(CHANNELS), 'channel order mismatch'
assert LABEL_COL in df.columns, 'missing eyeDetection label column'
assert df.isna().sum().sum() == 0, 'unexpected NaNs in raw data'
class_counts = df[LABEL_COL].value_counts().to_dict()
assert class_counts == {0: 8257, 1: 6723}, f'class balance mismatch: {class_counts}'
class_counts

{0: 8257, 1: 6723}

## Write the SHA-256 manifest

In [4]:
with open(RAW_CSV, 'rb') as f:
    sha = hashlib.sha256(f.read()).hexdigest()

manifest = {
    'sha256': sha,
    'downloaded_at': datetime.now(timezone.utc).isoformat(),
    'source_url': 'https://archive.ics.uci.edu/static/public/264/eeg+eye+state.zip',
    'n_rows': int(df.shape[0]),
    'n_cols': int(df.shape[1]),
    'channels': list(CHANNELS),
    'label_column': LABEL_COL,
    'class_counts': {str(k): int(v) for k, v in class_counts.items()},
}
with open(MANIFEST, 'w') as f:
    json.dump(manifest, f, indent=2, sort_keys=True)
manifest

{'sha256': 'b8f42456e5c0c3b7d9744fd9e1fbae3aa542eaaf3eec7b3ebd5abdb89da1a7b8',
 'downloaded_at': '2026-05-28T07:59:36.655184+00:00',
 'source_url': 'https://archive.ics.uci.edu/static/public/264/eeg+eye+state.zip',
 'n_rows': 14980,
 'n_cols': 15,
 'channels': ['AF3',
  'F7',
  'F3',
  'FC5',
  'T7',
  'P7',
  'O1',
  'O2',
  'P8',
  'T8',
  'FC6',
  'F4',
  'F8',
  'AF4'],
 'label_column': 'eyeDetection',
 'class_counts': {'0': 8257, '1': 6723}}

Data loaded successfully. The 14,980 × 15 EEG eye-state dataset has
been written to `data/raw/eeg_eye_state.csv` along with a SHA-256
manifest. The label balance (0: 8257, 1: 6723) and channel order
match the UCI Machine Learning Repository entry for dataset #264.
Downstream code (the `scripts/preprocess.py` CLI and the
`notebooks/01_eda.ipynb` analysis) reads from this canonical CSV.
